**Name: Shubham Tiwari (C3-41)**


**Subject: NLP**

# Web Application for Multilingual Indian News Translation and Summarization

## Step 1: Install Required Libraries

In [5]:
!pip install newspaper3k lxml_html_clean
!pip install langdetect
!pip install transformers sentencepiece sacremoses
!pip install torch

## Step 2: Import Libraries

In [6]:
import re
import pandas as pd
import torch

from newspaper import Article
import newspaper

from langdetect import detect

from transformers import MarianMTModel, MarianTokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print('Libraries imported successfully')
print('GPU Available:', torch.cuda.is_available())

Libraries imported successfully
GPU Available: True


## Step 3: User Input — Text or URL

In [7]:

# ── Choose input type ──────────────────────────────────────────
print("What do you want to translate and summarize?")
print("1. I will type / paste the text directly")
print("2. I have a URL (news article link)")
choice = input("Enter 1 or 2 : ").strip()

article_data = {}

if choice == "1":
    print("Paste your text below and press Enter twice when done:")
    lines = []
    while True:
        line = input()
        if line == "":
            break
        lines.append(line)
    user_text = " ".join(lines).strip()
    article_data["title"] = "User Input Text"
    article_data["text"] = user_text
    article_data["url"] = "N/A"
    article_data["authors"] = "N/A"
    article_data["publish_date"] = "N/A"
    article_data["source"] = "Manual Input"
    print(f"Text received ({len(user_text.split())} words)")

elif choice == "2":
    user_url = input("Paste the article URL : ").strip()
    print(f"Fetching article from: {user_url}")
    article = Article(user_url)
    article.download()
    article.parse()
    article_data["title"] = article.title
    article_data["text"] = article.text
    article_data["url"] = user_url
    article_data["authors"] = ", ".join(article.authors)
    article_data["publish_date"] = str(article.publish_date)
    article_data["source"] = user_url
    print(f"Title   : {article_data['title']}")
    print(f"Authors : {article_data['authors']}")
    print(f"Date    : {article_data['publish_date']}")
    print(f"Text    : {article_data['text'][:200]}...")

else:
    print("Invalid choice. Please run this cell again and enter 1 or 2.")


What do you want to translate and summarize?
1. I will type / paste the text directly
2. I have a URL (news article link)
Enter 1 or 2 : 1
Paste your text below and press Enter twice when done:
Two Australian states will offer free public transport to incentivise people not to drive as fuel prices soar due to the war in the Middle East.  Victoria, home to Melbourne, has said it will have free travel throughout April, while Tasmania has said commuters will not need to pay until the end of June. Other state governments have so far declined to follow suit.  It comes as the federal government announced it would halve the nation's fuel excise tax for three months to ease pressure on motorists' wallets.  Australia is among a host of nations that have seen fuel prices increase sharply since the start of the US-Israel war with Iran and the effective closure of the Strait of Hormuz.

Text received (120 words)


## Step 4: Language Detection

In [8]:
lang_names = {
    'hi': 'Hindi',
    'ta': 'Tamil',
    'te': 'Telugu',
    'bn': 'Bengali',
    'mr': 'Marathi',
    'kn': 'Kannada',
    'ml': 'Malayalam',
    'gu': 'Gujarati',
    'pa': 'Punjabi',
    'ur': 'Urdu',
    'en': 'English'
}

try:
    code = detect(article_data["text"][:500])
    article_data["lang_code"] = code
    article_data["lang_name"] = lang_names.get(code, code.upper())
except:
    article_data["lang_code"] = "en"
    article_data["lang_name"] = "English"

print(f"Detected Language : {article_data['lang_name']} ({article_data['lang_code']})")


Detected Language : English (en)


## Step 5: Text Preprocessing

In [9]:
def preprocess_text(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text)   # remove URLs
    text = re.sub(r'<[^>]+>', '', text)                  # remove HTML tags
    text = re.sub(r'\s+', ' ', text).strip()             # remove extra spaces
    return text

article_data["clean_text"] = preprocess_text(article_data["text"])

print('Preprocessing done')
print('\nCleaned text (first 300 chars):')
print(article_data["clean_text"][:300])


Preprocessing done

Cleaned text (first 300 chars):
Two Australian states will offer free public transport to incentivise people not to drive as fuel prices soar due to the war in the Middle East. Victoria, home to Melbourne, has said it will have free travel throughout April, while Tasmania has said commuters will not need to pay until the end of Ju


## Step 6: Translation using MarianMT (Local Model, No API)

In [10]:
translation_models = {
    'hi': 'Helsinki-NLP/opus-mt-hi-en',
    'ta': 'Helsinki-NLP/opus-mt-dra-en',
    'te': 'Helsinki-NLP/opus-mt-dra-en',
    'kn': 'Helsinki-NLP/opus-mt-dra-en',
    'ml': 'Helsinki-NLP/opus-mt-dra-en',
    'bn': 'Helsinki-NLP/opus-mt-bn-en',
    'mr': 'Helsinki-NLP/opus-mt-hi-en',
    'gu': 'Helsinki-NLP/opus-mt-gu-en',
    'ur': 'Helsinki-NLP/opus-mt-ur-en'
}

model_cache = {}

def translate_to_english(text, lang_code):
    if lang_code == "en":
        return text
    if lang_code not in translation_models:
        print(f"No translation model for: {lang_code}")
        return text
    model_name = translation_models[lang_code]
    if model_name not in model_cache:
        print(f"Loading model: {model_name}")
        tokenizer = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(model_name)
        model_cache[model_name] = (tokenizer, model)
    tokenizer, model = model_cache[model_name]
    inputs = tokenizer([text[:512]], return_tensors="pt", padding=True, truncation=True)
    outputs = model.generate(**inputs, num_beams=4, early_stopping=True)
    translated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated

print(f"Translating from {article_data['lang_name']}...")
article_data["translated_text"] = translate_to_english(article_data["clean_text"], article_data["lang_code"])
print('\nTranslated Text (first 400 chars):')
print(article_data["translated_text"][:400])


Translating from English...

Translated Text (first 400 chars):
Two Australian states will offer free public transport to incentivise people not to drive as fuel prices soar due to the war in the Middle East. Victoria, home to Melbourne, has said it will have free travel throughout April, while Tasmania has said commuters will not need to pay until the end of June. Other state governments have so far declined to follow suit. It comes as the federal government 


## Step 7: Summarization using T5 (Local Model, No API)

In [11]:
print('Loading T5 summarization model...')
sum_tokenizer = AutoTokenizer.from_pretrained('t5-small')
sum_model = AutoModelForSeq2SeqLM.from_pretrained('t5-small')
print('T5 model loaded')


Loading T5 summarization model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

OSError: Can't load tokenizer for 't5-small'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 't5-small' is the correct path to a directory containing all relevant files for a T5Tokenizer tokenizer.

In [ ]:
def summarize_text(text):
    if len(text.split()) < 30:
        return text
    input_text = "summarize: " + text
    inputs = sum_tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    summary_ids = sum_model.generate(
        inputs["input_ids"],
        num_beams=4,
        min_length=30,
        max_length=100,
        early_stopping=True
    )
    summary = sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

print('Summarizing...')
article_data["summary"] = summarize_text(article_data["translated_text"])
print('\nSummary:')
print(article_data["summary"])


## Step 8: Final Output

In [ ]:
print("=" * 60)
print(f"Title    : {article_data['title']}")
print(f"Language : {article_data['lang_name']}")
print(f"Source   : {article_data['source']}")
print(f"Authors  : {article_data['authors']}")
print(f"Date     : {article_data['publish_date']}")
print()
print("--- Original / Input Text ---")
print(article_data["clean_text"][:400])
print()
print("--- Translated Text (English) ---")
print(article_data["translated_text"][:400])
print()
print("--- Summary ---")
print(article_data["summary"])
print("=" * 60)


## Step 9: Save as DataFrame

In [ ]:
import pandas as pd

df = pd.DataFrame([{
    "Title"          : article_data["title"],
    "Language"       : article_data["lang_name"],
    "Authors"        : article_data["authors"],
    "Date"           : article_data["publish_date"],
    "Original Text"  : article_data["clean_text"][:300] + "...",
    "Translated Text": article_data["translated_text"][:300] + "...",
    "Summary"        : article_data["summary"],
    "URL"            : article_data["url"]
}])

df


## Step 10: Save Results to CSV and Download

In [ ]:
df.to_csv('news_result.csv', index=False)
print('Saved to news_result.csv')

from google.colab import files
files.download('news_result.csv')


In [ ]:
import gradio as gr
import torch


device = "cuda" if torch.cuda.is_available() else "cpu"

lang_names = {
    'hi': 'Hindi', 'ta': 'Tamil', 'te': 'Telugu', 'bn': 'Bengali',
    'mr': 'Marathi', 'kn': 'Kannada', 'ml': 'Malayalam',
    'gu': 'Gujarati', 'pa': 'Punjabi', 'ur': 'Urdu', 'en': 'English'
}

translation_models = {
    'hi': 'Helsinki-NLP/opus-mt-hi-en',
    'ta': 'Helsinki-NLP/opus-mt-dra-en',
    'te': 'Helsinki-NLP/opus-mt-dra-en',
    'kn': 'Helsinki-NLP/opus-mt-dra-en',
    'ml': 'Helsinki-NLP/opus-mt-dra-en',
    'bn': 'Helsinki-NLP/opus-mt-bn-en',
    'mr': 'Helsinki-NLP/opus-mt-hi-en',
    'gu': 'Helsinki-NLP/opus-mt-gu-en',
    'ur': 'Helsinki-NLP/opus-mt-ur-en'
}

model_cache = {}


sum_tokenizer = AutoTokenizer.from_pretrained('t5-small')
sum_model = AutoModelForSeq2SeqLM.from_pretrained('t5-small').to(device)


def preprocess_text(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def detect_language(text):
    try:
        code = detect(text[:500])
    except:
        code = "en"
    return code, lang_names.get(code, code.upper())


def translate_to_english(text, lang_code):
    if lang_code == "en":
        return text

    if lang_code not in translation_models:
        return text

    model_name = translation_models[lang_code]

    if model_name not in model_cache:
        tokenizer = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(model_name).to(device)
        model_cache[model_name] = (tokenizer, model)

    tokenizer, model = model_cache[model_name]

    inputs = tokenizer(
        [text[:512]],
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    outputs = model.generate(**inputs, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def summarize_text(text):
    if len(text.split()) < 30:
        return text

    inputs = sum_tokenizer(
        "summarize: " + text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    summary_ids = sum_model.generate(
        inputs["input_ids"],
        num_beams=4,
        min_length=30,
        max_length=100
    )

    return sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)


def process_text(input_text):
    clean = preprocess_text(input_text)

    lang_code, lang_name = detect_language(clean)
    translated = translate_to_english(clean, lang_code)
    summary = summarize_text(translated)

    df = pd.DataFrame([{
        "Language": lang_name,
        "Original Text": clean[:300] + "...",
        "Translated": translated[:300] + "...",
        "Summary": summary
    }])

    file_path = "result.csv"
    df.to_csv(file_path, index=False)

    return clean, translated, summary, lang_name, file_path


def process_url(url):
    article = Article(url)
    article.download()
    article.parse()

    return process_text(article.text)


with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as app:

    gr.Markdown("""
    # 📰 AI News Translator & Summarizer
    ### 🚀 Paste text OR enter URL → Get translation + summary instantly
    ### Tanish Agrawal (C3-50) & Shubham Tiwari (C3-41)
    """)

    with gr.Tabs():

        with gr.Tab("📝 Paste Text"):
            text_input = gr.Textbox(
                lines=10,
                placeholder="Paste your text here..."
            )
            btn_text = gr.Button("✨ Process Text")

        with gr.Tab("🌐 From URL"):
            url_input = gr.Textbox(
                placeholder="Paste news article URL..."
            )
            btn_url = gr.Button("🚀 Fetch & Process")

    gr.Markdown("## 📊 Results")

    language = gr.Textbox(label="Detected Language")
    original = gr.Textbox(label="Cleaned Text")
    translated = gr.Textbox(label="Translated Text (English)")
    summary = gr.Textbox(label="Summary")

    file_output = gr.File(label="Download CSV")


    btn_text.click(
        process_text,
        inputs=text_input,
        outputs=[original, translated, summary, language, file_output]
    )

    btn_url.click(
        process_url,
        inputs=url_input,
        outputs=[original, translated, summary, language, file_output]
    )

app.launch()

## Conclusion

In this project, I built a system for Multilingual Indian News Translation and Summarization.
The user can either paste text directly or provide a URL to any news article.
The system uses langdetect for language detection, MarianMT for local translation without any external API,
and T5 for abstractive text summarization.
This project demonstrates the application of NLP concepts like tokenization, sequence-to-sequence learning,
and attention mechanisms to solve real-world language barriers across Indian regional news.
